In [1]:
import pandas as pd
import numpy as np
import json

import torch
import torch.nn as nn
import torch.nn.functional as F

!pip install torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.5 MB/s eta 0:00:00


In [2]:
from torch_geometric.nn import GCNConv
from torch_geometric.utils import add_self_loops

In [4]:
edges_df= pd.read_csv("/content/edges.csv")
with open("/content/features.json", "r") as f:
  raw_features= json.load(f)

targets_df= pd.read_csv("/content/target.csv")

In [5]:
edges_df.head()

,id_1,id_2
0,0,23977
1,1,34526
2,1,2370
3,1,14683
4,1,29982


In [6]:
targets_df.head()

,id,name,ml_target
0,0,Eiryyy,0
1,1,shawflying,0
2,2,JpMCarrilho,1
3,3,SuhwanCha,0
4,4,sunilangadi2,1


In [7]:
len(raw_features)

37700

In [8]:
node_ids= set();

for _, row in edges_df.iterrows():
  node_ids.add(int(row["id_1"]))
  node_ids.add(int(row["id_2"]))


for nid in raw_features.keys():
  node_ids.add(int(nid))

for nid in targets_df["id"]:
  node_ids.add(int(nid))

node_ids= sorted(node_ids)

id2idx= {nid:i for i, nid in enumerate(node_ids)}
idx2id= {i:nid for nod, i in id2idx.items()}





In [9]:
len(node_ids)

37700

In [10]:
#raw_features

In [11]:
# feature encoding

feature_set= set()

for feat in raw_features.values():
  feature_set.update(feat)

feat2idx= {f: i for i, f in enumerate(sorted(feature_set))}


In [12]:
len(feat2idx)

4005

In [13]:
#feature matrix: each row is one node and col is one feat
N= len(node_ids)
f_dim= len(feat2idx)

X= torch.zeros((N, f_dim), dtype= torch.float)

for node_id_str, feat in raw_features.items():
  node_idx= id2idx[int(node_id_str)]
  for f in feat:
    X[node_idx, feat2idx[f]]= 1



In [14]:
edge_index = []

for _, row in edges_df.iterrows():
    u= id2idx[int(row["id_1"])]
    v= id2idx[int(row["id_2"])]

    edge_index.append([u, v])
    edge_index.append([v, u])  # undirected

edge_index= torch.tensor(edge_index, dtype=torch.long).t()

# // selfloops
edge_index, _= add_self_loops(edge_index, num_nodes=N)


In [20]:
# train, validate, test
y= torch.full((N,), -1, dtype=torch.long)

for _, row in targets_df.iterrows():
    y[id2idx[int(row["id"])]] = int(row["ml_target"])


In [ ]:
labeled_idx= (y!=-1).nonzero(as_tuple= True)[0]; # //node with labels
perm= torch.randperm(len(labeled_idx)) #shuffle
#split ration 75 15 15
train_end= int(0.7* len(perm))
cal_end= int(0.85* len(perm))

train_mask= torch.zeros(N, dtype=torch.bool)
val_mask= torch.zeros(N, dtype=torch.bool)
test_mask= torch.zeros(N, dtype=torch.bool)

train_mask[labeled_idx[perm[:train_end]]]= True

val_mask[labeled_idx[perm[train_end:cal_end]]]= True

test_mask[labeled_idx[perm[cal_end:]]]= True

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, 1)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        x = self.conv2(x, edge_index)
        return torch.sigmoid(x).squeeze()


In [ ]:
model= GCN(f_dim, 128)
optimizer= torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
loss_fn= nn.BCELoss()


In [22]:
for epoch in range(500):
    model.train()
    optimizer.zero_grad()

    out = model(X, edge_index)
    loss = loss_fn(out[train_mask], y[train_mask].float())

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(out[val_mask], y[val_mask].float())
        print(f"Epoch {epoch:03d} | Train {loss:.4f} | Val {val_loss:.4f}")


Epoch 000 | Train 0.3051 | Val 0.3294
Epoch 020 | Train 0.3026 | Val 0.3321
Epoch 040 | Train 0.3058 | Val 0.3302
Epoch 060 | Train 0.3033 | Val 0.3280
Epoch 080 | Train 0.3154 | Val 0.3361
Epoch 100 | Train 0.3017 | Val 0.3301
Epoch 120 | Train 0.2999 | Val 0.3304
Epoch 140 | Train 0.2985 | Val 0.3297
Epoch 160 | Train 0.2987 | Val 0.3337
Epoch 180 | Train 0.2982 | Val 0.3353
Epoch 200 | Train 0.3001 | Val 0.3307
Epoch 220 | Train 0.2959 | Val 0.3289
Epoch 240 | Train 0.2960 | Val 0.3277
Epoch 260 | Train 0.2969 | Val 0.3276
Epoch 280 | Train 0.2992 | Val 0.3367
Epoch 300 | Train 0.2950 | Val 0.3293
Epoch 320 | Train 0.2972 | Val 0.3279
Epoch 340 | Train 0.2945 | Val 0.3303
Epoch 360 | Train 0.2933 | Val 0.3300
Epoch 380 | Train 0.2954 | Val 0.3347
Epoch 400 | Train 0.2937 | Val 0.3336
Epoch 420 | Train 0.2928 | Val 0.3292
Epoch 440 | Train 0.2934 | Val 0.3328
Epoch 460 | Train 0.2966 | Val 0.3390
Epoch 480 | Train 0.2950 | Val 0.3369


In [23]:
model.eval()
with torch.no_grad():
    preds = (out[test_mask] > 0.5).long()
    acc = (preds == y[test_mask]).float().mean()

print("Test Accuracy:", acc.item())


Test Accuracy: 0.8714411854743958
